# Speculative Decoding in Production with **vLLM** (Companion Notebook)

Runnable companion to the blog post
[*Speculative Decoding in Production with vLLM*](https://YOUR_BLOG_DOMAIN/chapters/inference/speculative-decoding-part-1).

**How to use it:** open this inside your RunPod pod (JupyterLab) and run it top to bottom,
one cell at a time, reading the matching section in the blog as you go. Each code cell does
exactly one thing and runs the same command you see in the post.

> Run on the GPU box itself, not a laptop. The section numbers here (**1**, **2.1**, **4.2**) match the blog.

## 1. What we're building

We serve `meta-llama/Llama-3.1-8B-Instruct` on one 48 GB GPU and benchmark it twice: once plain (**baseline**), once
with **EAGLE3**. Same model, same prompts; we flip one flag and read the speedup.

## 2. Setup

### 2.1 Install vLLM

In [1]:
!pip install -q -U vllm openai


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


### 2.2 Log in to Hugging Face

In [ ]:
# Log in to Hugging Face (Llama is gated: accept the license on the model page first).
# Weights cache on the persistent volume so they survive a pod restart.
import os
os.environ["HF_HOME"] = "/workspace/hf"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

from huggingface_hub import login
login()   # paste your HF token when prompted

### 2.3 Settings

The one place to edit. `TP = 1` because 8B fits on a single 48 GB card (A40, L40S,
RTX 6000 Ada, or A6000), so there is no tensor parallelism and no quantization.

In [5]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
DRAFT = "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B"   # pre-trained EAGLE3 head for this model
TP    = 1                                    # one 48 GB card is enough for 8B

### 2.4 The server helper

In [6]:
# --- One small helper: start a server in the background and wait until it's healthy. ---
# `vllm serve` holds the terminal, so we can't
# just run it in a cell. This launches it in the background, then blocks until the server
# answers. `stop()` shuts it down. This is the only "framework" code in the notebook;
# every command it runs is exactly what the blog shows.
import subprocess, time, os, requests

def start(cmd, env=None, log="/workspace/server.log"):
    print("launching:", " ".join(cmd))
    p = subprocess.Popen(cmd, env={**os.environ, **(env or {})},
                         stdout=open(log, "w"), stderr=subprocess.STDOUT)
    url = "http://localhost:8000/health"
    while True:
        if p.poll() is not None:
            raise RuntimeError("server exited early, check " + log)
        try:
            if requests.get(url, timeout=5).status_code == 200:
                print("server is up on port 8000"); return p
        except Exception:
            pass
        time.sleep(5)

def stop(p):
    if p is None: return
    p.terminate()
    try: p.wait(60)
    except Exception: p.kill()
    time.sleep(5)
    print("server stopped")

## 3. Why EAGLE3

Speculative decoding uses a small, fast *drafter* to guess the next few tokens, and the big
model verifies them all in one pass, so several tokens can be accepted for the price of one
forward pass. **EAGLE3** is the drafter that won in production: instead of a whole second
model, it is a tiny *head* bolted onto the target that predicts the target's own internal
features. That means the highest acceptance rates, almost no extra VRAM, no tokenizer to
match, and pre-trained heads already exist for `meta-llama/Llama-3.1-8B-Instruct`, so there is nothing to train.

Everything below follows one simple loop for each configuration:

> **start the server → benchmark it → stop the server**

Only one server can hold the GPU at a time, so we always stop one before starting the next.
We run that loop twice: once for the plain model (the **baseline**), once with **EAGLE3** on.
The ratio between the two is the speedup.

## 4. vLLM implementation

### 4.1 Baseline (no speculation)

First we measure the model **as-is**, with no speculation. This is the reference every speedup
is compared against, so we always run it first. The cell below launches a normal vLLM server
(the exact command from blog §4.1): `-tp 1` puts the whole model on one GPU, and
`--gpu-memory-utilization 0.90` lets vLLM use most of the card for the KV cache. It takes a
minute or two to load; the helper waits until it is ready.

In [7]:
baseline = start(
    ["vllm", "serve", MODEL, "--seed", "42", "-tp", str(TP),
     "--gpu-memory-utilization", "0.90"],
    env={"VLLM_USE_V1": "1"})

launching: vllm serve meta-llama/Llama-3.1-8B-Instruct --seed 42 -tp 1 --gpu-memory-utilization 0.90
server is up on port 8000


The server is up. Run the quick speed check and **write down the tok/s** — this is your baseline number:

In [8]:
# Quick speed check: send ONE request, stream the answer, and measure tokens per second.
# This is the simplest possible speedup meter. Run it once now (baseline) and again after
# EAGLE3 is on, then compare the two tok/s numbers.
import time, openai
client = openai.Client(base_url="http://localhost:8000/v1", api_key="None")

t0, n = time.time(), 0
stream = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Write a 400-word explanation of how a CPU cache works."}],
    temperature=0, max_tokens=512, stream=True)
for ch in stream:
    if ch.choices[0].delta.content:
        n += 1
dt = time.time() - t0
print(f"{n} tokens in {dt:.1f}s = {n/dt:.1f} tok/s")

512 tokens in 11.2s = 45.7 tok/s


**Optional: rigorous numbers (blog §7.1).** The quick check above is one request. For real
numbers under load, vLLM ships its own benchmark that fires 500 chat prompts at the server and
reports throughput and latency percentiles. It also prints **acceptance length** for EAGLE3
runs, the average number of drafted tokens accepted per step, which is the single best sign of
whether speculation is actually working. First download the ShareGPT chat dataset (once), then
run the benchmark. Skip this if you only want the quick tok/s comparison.

In [9]:
!wget -q -O /workspace/sharegpt.json \
  https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json

In [10]:
!vllm bench serve --backend vllm --model meta-llama/Llama-3.1-8B-Instruct --endpoint /v1/completions \
  --dataset-name sharegpt --dataset-path /workspace/sharegpt.json \
  --num-prompts 500 --request-rate inf

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7f4fd9f86e80>, trust_remote_code=False, seed=0, num_prompts=500, dataset_name='sharegpt', no_stream=False, dataset_path='/workspace/sharegpt.json', no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix

Stop the baseline server before starting the next one (only one can hold the GPU):

In [11]:
stop(baseline)

server stopped


### 4.2 EAGLE3

Now the interesting run. It is the **same server command** with one extra argument:
`--speculative-config`. That JSON tells vLLM to attach the EAGLE3 draft head (`method:
"eagle3"`, our pre-trained `DRAFT`) and to draft `num_speculative_tokens: 3` tokens ahead each
step. That token count is the one dial worth tuning (the lookahead); vLLM sizes the draft tree
from it for you. Nothing else changes: the model, the prompts, and the output stay identical.
That is the whole point of speculative decoding, same answers, ideally at more tokens per
second (though whether you actually get them is what we measure). (Exact blog §4.2 command.)

In [12]:
import json
spec = json.dumps({"method": "eagle3", "model": DRAFT, "num_speculative_tokens": 3})

eagle3 = start(
    ["vllm", "serve", MODEL, "--seed", "42", "-tp", str(TP),
     "--speculative-config", spec],
    env={"VLLM_USE_V1": "1"})

launching: vllm serve meta-llama/Llama-3.1-8B-Instruct --seed 42 -tp 1 --speculative-config {"method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 3}
server is up on port 8000


Run the **same** quick speed check. Compare this tok/s to the baseline you wrote down; the jump is your EAGLE3 speedup:

In [13]:
# Quick speed check: send ONE request, stream the answer, and measure tokens per second.
# This is the simplest possible speedup meter. Run it once now (baseline) and again after
# EAGLE3 is on, then compare the two tok/s numbers.
import time, openai
client = openai.Client(base_url="http://localhost:8000/v1", api_key="None")

t0, n = time.time(), 0
stream = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[{"role": "user", "content": "Write a 400-word explanation of how a CPU cache works."}],
    temperature=0, max_tokens=512, stream=True)
for ch in stream:
    if ch.choices[0].delta.content:
        n += 1
dt = time.time() - t0
print(f"{n} tokens in {dt:.1f}s = {n/dt:.1f} tok/s")

184 tokens in 4.7s = 39.1 tok/s


**Optional:** the identical benchmark, run while the EAGLE3 server is up. It prints throughput and latency below.

In [ ]:
!vllm bench serve --backend vllm --model meta-llama/Llama-3.1-8B-Instruct --endpoint /v1/completions \
  --dataset-name sharegpt --dataset-path /workspace/sharegpt.json \
  --num-prompts 500 --request-rate inf

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7f3db6196e80>, trust_remote_code=False, seed=0, num_prompts=500, dataset_name='sharegpt', no_stream=False, dataset_path='/workspace/sharegpt.json', no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=1024, random_output_len=128, random_range_ratio='0.0', random_prefix

vLLM logs the **acceptance length** (the average tokens accepted per step) to the server log
rather than the benchmark output, so pull it out here. This number only exists for the EAGLE3
run; ~3-4 means speculation is working well, ~1 means it is not.

In [ ]:
!grep -iE "accept|draft|spec" /workspace/server.log | tail -15

In [ ]:
stop(eagle3)

### 4.3 Offline (Python) alternative

In-process generation instead of a server. **Restart the kernel first** (the cells above hold
the GPU). Matches blog §4.3.

In [ ]:
# from vllm import LLM, SamplingParams
# llm = LLM(model=MODEL, tensor_parallel_size=TP,
#           speculative_config={"method": "eagle3", "model": DRAFT,
#                               "num_speculative_tokens": 3})
# print(llm.generate(["Explain speculative decoding in one paragraph."],
#                    SamplingParams(temperature=0, max_tokens=256))[0].outputs[0].text)

## 8. Results

Here is what this run actually produced (Llama-3.1-8B, single 48 GB Ada GPU). Reporting it straight, because the honest result teaches more than a flattering one: **EAGLE3 came out slower in both regimes.**

| Test | Baseline | EAGLE3 | vs baseline |
| --- | --- | --- | --- |
| Single request (greedy, `temperature=0`) | 45.7 tok/s | 39.1 tok/s | **0.86×** |
| Under load (500 prompts, `--request-rate inf`) | 2589 tok/s | 2103 tok/s | **0.81×** |

**Acceptance length $\tau = 1.81$** (rate 27%, per-position 44% / 23% / 14%). That is the smoking gun: barely above 1, far below the 3-4 you want. When fewer than two of every three drafted tokens survive verification, the draft-and-verify overhead is not paid back, so throughput drops.

Why acceptance was low, and why this is a normal outcome:

1. **Small target on a fast GPU is the hard case.** An 8B already decodes quickly (45 tok/s single-stream), so there is little memory-bandwidth headroom for speculation to buy back, and the draft head's own forward pass eats most of any gain. EAGLE3's economics are far better on a **70B** target.
2. **The load benchmark ran non-greedy** (vLLM no longer forces `temperature=0`), which lowers acceptance.
3. **`--request-rate inf` saturated the GPU** (~500 concurrent), so speculation's extra work stole cycles from useful generation.

This is the "speculative decoding is a bet" lesson from Part 1, live.

## 9. When speculation helps, and when it hurts

**Helps:** decode-heavy chat/reasoning traffic, well-aligned drafts, low-to-medium load.
**Hurts or does nothing:** prefill-heavy prompts, or a GPU already saturated by a big batch.
A rejection still triggers a serial fallback, so acceptance length is the number to watch.

## 10. Takeaways

- **8B is a single-GPU job** on one 48 GB card: `-tp 1`, no quantization.
- **EAGLE3 is one flag** (`--speculative-config`), not a rewrite.
- **Read acceptance length first.** It tells you whether EAGLE3 is working before throughput does. Ours ($\tau = 1.81$) predicted the slowdown in one glance.
- **It is conditional, and it can lose.** A small model on a fast, saturated GPU is the hard case, and ours came out slower. The win grows with model size and acceptance.

> Serve, turn on EAGLE3, benchmark, read $\tau$. If $\tau$ is near 1, it is not helping you.

## 11. Insight from this run

The one thing to carry away: **acceptance length is the number that matters, and here it told the whole story before any throughput figure did.** $\tau = 1.81$ meant EAGLE3 was never going to win on this setup.

- Both tests lost for related reasons: modest acceptance could not overcome the draft overhead (single request, 39.1 vs 45.7), and at 500 concurrent the GPU had no spare compute for the extra drafting (2103 vs 2589).
- On a **small model + fast single GPU**, EAGLE3 is a close call and can go negative. The win grows with **model size** (70B is where it shines) and with **acceptance**.
- **To try for a win on this box:** run the latency-mode benchmark (`vllm bench serve ... --max-concurrency 1 --temperature 0`), try `num_speculative_tokens: 2` (shorter drafts fail less often), and confirm the draft head actually attached (the acceptance line in `/workspace/server.log`).
- A negative result is still a result. It is the honest version of the whole point of Part 1: speculative decoding is a bet, and you measure it, you do not assume it.